# RiskGuard AI — Phase 1: PaySim Data Exploration & Leakage Analysis

**Objective**: Conduct an empirical, rigorous exploration of the PaySim synthetic financial transaction dataset, characterize class imbalance, identify fraud patterns, and perform a strict data leakage audit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("../data/raw/PS_20174392719_1491204439457_log.csv")
print(f"Data path exists: {DATA_PATH.exists()}")

## 1. Dataset Shape, Columns, and Data Types

In [ ]:
dtypes = {
    'step': 'int32',
    'type': 'category',
    'amount': 'float32',
    'nameOrig': 'object',
    'oldbalanceOrg': 'float32',
    'newbalanceOrig': 'float32',
    'nameDest': 'object',
    'oldbalanceDest': 'float32',
    'newbalanceDest': 'float32',
    'isFraud': 'int8',
    'isFlaggedFraud': 'int8'
}
df = pd.read_csv(DATA_PATH, dtype=dtypes)
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.info()

## 2. Missing Values & Duplicate Check

In [ ]:
print("--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
dup_count = df.duplicated().sum()
print(f"Exact duplicates: {dup_count}")

## 3. Fraud Distribution (Extreme Class Imbalance)

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].mean() * 100
print(f"Legitimate transactions (0): {fraud_counts[0]:,} ({100 - fraud_pct:.3f}%)")
print(f"Fraudulent transactions (1): {fraud_counts[1]:,} ({fraud_pct:.3f}%)")
print(f"Imbalance ratio: 1 fraud per {fraud_counts[0] // fraud_counts[1]:,} legitimate transactions")

## 4. Transaction Types & Fraud by Type

**Key Finding**: Fraud in PaySim occurs **EXCLUSIVELY** in `TRANSFER` and `CASH_OUT` transactions. `PAYMENT`, `CASH_IN`, and `DEBIT` contain exactly zero fraudulent instances.

In [ ]:
type_summary = df.groupby('type', observed=False)['isFraud'].agg(total='count', fraud_count='sum', fraud_rate='mean')
type_summary['fraud_rate_pct'] = type_summary['fraud_rate'] * 100
print(type_summary)

## 5. Amount Distribution (Legit vs Fraud)

In [ ]:
print("--- Overall Amount Statistics ---")
print(df['amount'].describe().apply(lambda x: f"{x:,.2f}"))

print("\n--- Fraudulent Amount Statistics ---")
print(df[df['isFraud'] == 1]['amount'].describe().apply(lambda x: f"{x:,.2f}"))

print("\n--- Legitimate Amount Statistics ---")
print(df[df['isFraud'] == 0]['amount'].describe().apply(lambda x: f"{x:,.2f}"))

## 6. Temporal Distribution

- 1 step = 1 hour
- Step range: 1 to 743 (~31 days)
- Fraud occurs across the entire simulation duration.

In [ ]:
print(f"Step min: {df['step'].min()}, max: {df['step'].max()}")
fraud_steps = df[df['isFraud'] == 1]['step']
print(f"Fraud step min: {fraud_steps.min()}, max: {fraud_steps.max()}")

## 7. Critical Data Leakage Audit

### Excluded Features & Justifications:

1. **`isFlaggedFraud`**:
   - This is a simulation heuristic (flags `TRANSFER` > 200,000).
   - In the dataset, it caught only 16 out of 8,213 frauds (0.19% recall) and has 0 false positives.
   - Including it creates direct target leakage / simulator rule coupling without representing actual model discriminative capability.

2. **`nameOrig` and `nameDest` (Raw Account Identifiers)**:
   - `nameOrig` has >6.35M unique IDs for 6.36M transactions (transacted only once).
   - Raw strings cannot generalize to unseen customer IDs in real deployment and lead to high-cardinality overfitting or synthetic ID memorization.

3. **`newbalanceOrig` & `newbalanceDest` (Post-Transaction State Leakage)**:
   - Real-world fraud prevention systems must evaluate transactions **pre-authorization** before the transaction settles.
   - In PaySim, fraudulent transactions artificially zero out `newbalanceOrig` (draining the entire account) in >99% of cases where `oldbalanceOrg > 0`.
   - Relying on `newbalanceOrig == 0` directly leaks the post-transaction consequence of an account drain.
   - Similarly, `newbalanceDest` is a post-settlement outcome and suffers from simulation artifacts (many destinations remain 0 despite incoming funds).

In [ ]:
# Empirical demonstration of balance draining in fraud
fraud_txns = df[df['isFraud'] == 1]
drain_mask = (fraud_txns['oldbalanceOrg'] > 0) & (fraud_txns['newbalanceOrig'] == 0)
print(f"Fraud cases where account is fully drained (old > 0, new == 0): {drain_mask.mean()*100:.2f}%")

amt_eq_bal = (fraud_txns['amount'] == fraud_txns['oldbalanceOrg']).mean()
print(f"Fraud cases where exact full balance is transferred (amount == oldbalanceOrg): {amt_eq_bal*100:.2f}%")